In [3]:
# CELL 1 — SETUP AND BANGLA AGRICULTURE DATA
!pip install -q -U unsloth transformers trl datasets accelerate peft bitsandbytes huggingface_hub sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, re, gc, unicodedata, shutil
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Upload the three files to the Colab session (/content), or place them in this folder.
TRAIN_NAME = "Bangla_Agriculture_QA_Train_800.json"
TEST_NAME = "Bangla_Agriculture_QA_Test_200.json"
KB_NAME = "Bangla_Agriculture_Knowledge_Base_500.json"

def locate(name):
    candidates = [Path('/content')/name, Path('/content/drive/MyDrive')/name, Path.cwd()/name]
    for p in candidates:
        if p.exists(): return p
    matches = list(Path('/content').rglob(name)) if Path('/content').exists() else []
    if matches: return matches[0]
    raise FileNotFoundError(f"Upload or copy {name} to /content")

TRAIN_PATH, TEST_PATH, KB_PATH = map(locate, [TRAIN_NAME, TEST_NAME, KB_NAME])
# Change this only if you want another Drive location.
OUT = Path('/content/drive/MyDrive/Bangla_Agriculture_Finetune/MODEL_FOLDER')
MODEL_DIR = OUT / 'model_adapter'
OUT.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)

with open(TRAIN_PATH, encoding='utf-8') as f: train_raw = json.load(f)['qa_pairs']
with open(TEST_PATH, encoding='utf-8') as f: test_raw = json.load(f)['qa_pairs']
with open(KB_PATH, encoding='utf-8') as f: knowledge_base = json.load(f)

train_rows = []
for row in train_raw:
    q, a = str(row.get('question','')).strip(), str(row.get('reference_answer','')).strip()
    if q and a: train_rows.append({'id': str(row.get('id','')), 'question': q, 'answer': a})
tests = pd.DataFrame([{'id': str(r.get('id','')), 'question': str(r.get('question','')).strip(), 'gold': str(r.get('reference_answer','')).strip()} for r in test_raw])
assert len(train_rows) == 800 and len(tests) == 200 and tests['gold'].str.strip().ne('').all()
with open(OUT/'unified_train_used.json','w',encoding='utf-8') as f: json.dump(train_rows,f,ensure_ascii=False,indent=2)
print(f'Training QA: {len(train_rows)} | Test QA: {len(tests)} | Knowledge-base records: {len(knowledge_base)}')
print('Output folder:', OUT)


Mounted at /content/drive
Training QA: 800 | Test QA: 200 | Knowledge-base records: 500
Output folder: /content/drive/MyDrive/Bangla_Agriculture_Finetune/MODEL_FOLDER


In [4]:
# CELL 2 — QLORA FINE-TUNING (mistralai/Mistral-7B-Instruct-v0.3)
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login
notebook_login()

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_FOLDER = "Mistral_7B_Instruct_v0.3"
OUT = Path('/content/drive/MyDrive/Bangla_Agriculture_Finetune') / MODEL_FOLDER
MODEL_DIR = OUT / 'model_adapter'
MAX_SEQ_LENGTH = 1024
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=3407)
SYSTEM_PROMPT = "তুমি বাংলাদেশের কৃষি বিষয়ে একজন সহায়ক সহকারী। শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না।'"

def formatting(example):
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":example['question']},{"role":"assistant","content":example['answer']}]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset=Dataset.from_list(train_rows).map(formatting, remove_columns=['id','question','answer'])
trainer=SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, packing=True, args=SFTConfig(output_dir=str(OUT/'checkpoints'), num_train_epochs=5, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, optim='adamw_8bit', lr_scheduler_type='cosine', warmup_ratio=0.03, weight_decay=0.01, fp16=False, bf16=torch.cuda.is_bf16_supported(), logging_steps=10, save_strategy='epoch', save_total_limit=3, report_to='none', seed=3407))
train_result=trainer.train()
model.save_pretrained(str(MODEL_DIR)); tokenizer.save_pretrained(str(MODEL_DIR))
pd.DataFrame(trainer.state.log_history).to_csv(OUT/'training_log.csv',index=False)
config={'base_model':MODEL_NAME,'method':'QLoRA with Unsloth','epochs':5,'max_seq_length':MAX_SEQ_LENGTH,'lora_r':16,'lora_alpha':16,'learning_rate':2e-4,'batch_size':2,'gradient_accumulation_steps':4,'packing':True,'seed':3407}
with open(OUT/'training_config.json','w',encoding='utf-8') as f: json.dump(config,f,ensure_ascii=False,indent=2)
print('Fine-tuning complete. Adapter saved to', MODEL_DIR)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.528349
20,0.914670
30,0.718380
40,0.543797
50,0.455321
60,0.364007
70,0.322023
80,0.286905


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-16/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-16.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-32/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-32.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-48/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/checkpoint

Fine-tuning complete. Adapter saved to /content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/model_adapter


In [5]:
# CELL 3 — INFERENCE ON THE 200 QUESTION TEST SET
from tqdm.auto import tqdm
FastLanguageModel.for_inference(model); tokenizer.padding_side='left'; MAX_NEW_TOKENS=500
OUT = Path('/content/drive/MyDrive/Bangla_Agriculture_Finetune') / 'Mistral_7B_Instruct_v0.3'
PARTIAL=OUT/'predictions_partial.csv'
def generate_answer(question):
    messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':str(question)}]
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=4096).to('cuda')
    with torch.no_grad(): outputs=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.05,use_cache=True,pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id)
    generated=outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated,skip_special_tokens=True).strip(), (len(generated)>=MAX_NEW_TOKENS and generated[-1].item()!=tokenizer.eos_token_id)
done={}
if PARTIAL.exists():
    for _,r in pd.read_csv(PARTIAL).fillna('').iterrows(): done[str(r['id'])]=r.to_dict()
predictions=[]
for _,row in tqdm(tests.iterrows(),total=len(tests),desc='Fine-tuned model'):
    key=str(row['id'])
    result=done.get(key)
    if result is None:
        ans,tr=generate_answer(row['question']); result={'id':key,'question':row['question'],'gold':row['gold'],'prediction':ans,'truncated':tr}
    predictions.append(result); pd.DataFrame(predictions).to_csv(PARTIAL,index=False,encoding='utf-8-sig')
pd.DataFrame(predictions).to_csv(OUT/'predictions.csv',index=False,encoding='utf-8-sig')
print('Completed:',len(predictions)); del model; gc.collect(); torch.cuda.empty_cache()


Fine-tuned model:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Completed: 200


In [6]:
# ============================================================
# CELL 4 — EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision, Recall and F1
# ============================================================

from collections import Counter

from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all(), "Blank gold answer found."


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ------------------------------------------------------------
# Exact Match
# ------------------------------------------------------------

def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ------------------------------------------------------------
# Token F1
# ------------------------------------------------------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-N
# ------------------------------------------------------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p) - n + 1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g) - n + 1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(
                    dp[j - 1] + 1
                )

            else:
                new.append(
                    max(
                        dp[j],
                        new[-1]
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# Row metrics
# ------------------------------------------------------------

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]


# ------------------------------------------------------------
# Corpus BLEU
# ------------------------------------------------------------

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ------------------------------------------------------------
# BERTScore
# ------------------------------------------------------------

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),

    model_type="bert-base-multilingual-cased",

    batch_size=4,
    device="cpu",

    idf=False,
    rescale_with_baseline=False,

    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ------------------------------------------------------------
# Overall result table
# ------------------------------------------------------------

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


# Save predictions with row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall result
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved artifacts:")
print(OUT / "model_adapter")
print(OUT / "unified_train_used.json")
print(OUT / "training_config.json")
print(OUT / "training_log.csv")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/50 [00:00<?, ?it/s]

done in 23.26 seconds, 8.60 sentences/sec


,metric,score
0,Exact Match,0.000000
1,Fuzzy Match,0.529260
2,Corpus BLEU,0.038359
3,ROUGE-1,0.217909
4,ROUGE-2,0.072121
5,ROUGE-L,0.198658
6,Token F1,0.217909
7,BERT Precision,0.750394
8,BERT Recall,0.750475
9,BERT F1,0.749528



Saved artifacts:
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/model_adapter
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/unified_train_used.json
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/training_config.json
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/training_log.csv
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/predictions_partial.csv
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/predictions.csv
/content/drive/MyDrive/Bangla_Agriculture_Finetune/Mistral_7B_Instruct_v0.3/result.csv
